In [3]:
import json
import random
import time
import sys
from pathlib import Path
import pandas as pd
from tqdm import tqdm

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("✅ Project root:", PROJECT_ROOT)
from agents.orchestrator import reflex_orchestrator
from agents.verifier import execution_verifier


✅ Project root: C:\Users\sprdh\Downloads\Axiom-SQL-Reflex-V4
🔥 cartographer.py LOADED


In [7]:
SPIDER_DEV = Path("../data/spider/dev.json")

with open(SPIDER_DEV) as f:
    spider = json.load(f)

DB_ID = "student_transcripts_tracking"  

samples = [s for s in spider if s["db_id"] == DB_ID]

print("Total questions:", len(samples))


Total questions: 78


In [8]:
random.seed(42)
SAMPLE_SIZE = min(100, len(samples))
eval_samples = random.sample(samples, SAMPLE_SIZE)

print("Evaluating:", SAMPLE_SIZE)


Evaluating: 78


In [9]:
from agents.cartographer import (
    build_schema_graph,
    build_df,
    build_faiss_index,
    graphrag_cartographer
)

DB_PATH = Path(f"../data/spider/database/{DB_ID}/{DB_ID}.sqlite")

graph, schema_texts, schema_ids, doc_tokens = build_schema_graph(DB_PATH)
df_stats = build_df(doc_tokens)
embedder, faiss_index = build_faiss_index(schema_texts)

print("Schema tables:", len(schema_ids))


c:\Users\sprdh\Downloads\Axiom-SQL-Reflex-V4\.venv\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Schema tables: 11


In [10]:
from llama_cpp import Llama

MODELS_DIR = Path("../models")

llm_large = Llama(
    model_path=str(MODELS_DIR / "deepseek-coder-6.7b-instruct.Q4_K_M.gguf"),
    n_ctx=2048, n_threads=8, n_batch=256
)

llm_medium = Llama(
    model_path=str(MODELS_DIR / "mistral-7b-instruct-v0.2.Q4_K_M.gguf"),
    n_ctx=2048, n_threads=8, n_batch=256
)

print("LLMs loaded")


AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 0 | VSX = 0 | 


LLMs loaded


AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 0 | VSX = 0 | 


In [11]:
from agents.architect import architect_ensemble
from agents.verifier import execution_verifier
def exec_result(con, sql):
    try:
        return con.execute(sql).fetchall()
    except Exception:
        return None
import duckdb

import duckdb

def evaluate_one(sample):
    question = sample["question"]
    gold_sql = sample["query"]

    con = duckdb.connect(str(DB_PATH))  # ✅ FIX

    gold_res = exec_result(con, gold_sql)
    if gold_res is None:
        return False, None, gold_sql

    carto = graphrag_cartographer(
        question=question,
        graph=graph,
        schema_texts=schema_texts,
        schema_ids=schema_ids,
        embedder=embedder,
        faiss_index=faiss_index,
        doc_tokens=doc_tokens,
        df=df_stats
    )

    candidates = architect_ensemble(
        question=question,
        schema="\n".join(schema_texts),
        llm_large=llm_large,
        llm_medium=llm_medium
    )

    for cand in candidates:
        sql = cand["sql"]
        pred_res = exec_result(con, sql)
        if pred_res == gold_res:
            return True, sql, gold_sql

    return False, None, gold_sql



In [12]:
correct = 0
records = []

start = time.time()

for i, s in enumerate(eval_samples, 1):
    ok, pred_sql, gold_sql = evaluate_one(s)

    correct += int(ok)
    acc = correct / i

    print(f"\n[{i}] Q:", s["question"])
    print("Pred:", pred_sql)
    print("Gold:", gold_sql)
    print("Result:", "✅ CORRECT" if ok else "❌ WRONG")
    print(f"Running accuracy: {acc:.2%}")

    records.append({
        "question": s["question"],
        "pred_sql": pred_sql,
        "gold_sql": gold_sql,
        "correct": ok
    })

elapsed = time.time() - start


Llama.generate: prefix-match hit



[1] Q: How many degrees does the engineering department offer?
Pred: SELECT COUNT(*) FROM Degree_Programs WHERE Department_ID = (SELECT Department_ID FROM Departments WHERE Department_Name = 'Engineering');
Gold: SELECT count(*) FROM Departments AS T1 JOIN Degree_Programs AS T2 ON T1.department_id  =  T2.department_id WHERE T1.department_name  =  'engineer'
Result: ✅ CORRECT
Running accuracy: 100.00%


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit



[2] Q: How many courses are there?
Pred: SELECT COUNT(DISTINCT Courses.course_id) FROM Courses;
Gold: SELECT count(*) FROM Courses
Result: ✅ CORRECT
Running accuracy: 100.00%

[3] Q: What is the first, middle, and last name, along with the id and number of enrollments, for the student who enrolled the most in any program?
Pred: None
Gold: SELECT T1.student_id ,  T1.first_name ,  T1.middle_name ,  T1.last_name ,  count(*) ,  T1.student_id FROM Students AS T1 JOIN Student_Enrolment AS T2 ON T1.student_id  =  T2.student_id GROUP BY T1.student_id ORDER BY count(*) DESC LIMIT 1
Result: ❌ WRONG
Running accuracy: 66.67%


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


...


[76] Q: Whose permanent address is different from his or her current address? List his or her first name.
Pred: SELECT first_name FROM Students WHERE current_address_id != permanent_address_id;
Gold: SELECT first_name FROM Students WHERE current_address_id != permanent_address_id
Result: ✅ CORRECT
Running accuracy: 36.84%


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit



[77] Q: Who is the first student to register? List the first name, middle name and last name.
Pred: SELECT first_name, middle_name, last_name FROM Students ORDER BY date_first_registered ASC LIMIT 1;
Gold: SELECT first_name ,  middle_name ,  last_name FROM Students ORDER BY date_first_registered ASC LIMIT 1
Result: ✅ CORRECT
Running accuracy: 37.66%


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit



[78] Q: What is the mobile phone number of the student named Timmothy Ward ?
Pred: None
Gold: select cell_mobile_number from students where first_name  =  'timmothy' and last_name  =  'ward'
Result: ❌ WRONG
Running accuracy: 37.18%


In [13]:
df = pd.DataFrame(records)

print("\nFINAL RESULTS")
print("Accuracy:", df["correct"].mean())
print("Avg time / query:", elapsed / len(df))



FINAL RESULTS
Accuracy: 0.3717948717948718
Avg time / query: 31.251658757527668
